# 最終課題

In [1]:
# --- 必要なライブラリをインポート ---
import requests # Webページを取得するため
from bs4 import BeautifulSoup # HTMLを解析するため
import time # 負荷軽減の sleep のため
from urllib.parse import urljoin, urlparse # URLを扱うため

# --- 基本設定 ---
START_URL = "https://www.musashino-u.ac.jp/"
BASE_DOMAIN = urlparse(START_URL).netloc # "www.musashino-u.ac.jp" を取得

# --- 課題の要件 ---
# 1. 最終的にデータを格納する辞書
sitemap_data = {}

# 2. これからアクセスするURLのリスト（※set型を使うと重複が自動で消えて効率的）
to_visit = set()
to_visit.add(START_URL) # スタート地点を追加

# 3. 既にアクセス済みのURLのリスト（無限ループを防ぐため）
visited = set()

print(f"クローリングを開始します。対象ドメイン: {BASE_DOMAIN}")

クローリングを開始します。対象ドメイン: www.musashino-u.ac.jp


In [4]:
# to_visit リストが空になるまで繰り返す
while to_visit:
    
    # 1. 次にアクセスするURLを取り出す
    current_url = to_visit.pop()

    # 2. もし既に訪問済みなら、この後の処理をスキップ
    if current_url in visited:
        continue

    # 3. 訪問済みに登録
    visited.add(current_url)
    # どのページを見ているか分かりやすくするためにprint
    print(f"[{len(visited)}] チェック中: {current_url}")

    # 4. 課題の要件：負荷軽減（アクセスする直前）
    # ※ 課題の指示にあるため、必ず入れます
    time.sleep(1) # 1秒待機

    # 5. Webページにアクセス
    try:
        response = requests.get(current_url, timeout=5)
        # もしHTTPエラー（404 Not Foundなど）なら、ここで処理を中断
        response.raise_for_status() 

        # --- ページの解析開始 ---
        
        soup = BeautifulSoup(response.text, 'html.parser')

        # 【課題要件】<title> を抽出
        if soup.title and soup.title.string:
            title_text = soup.title.string.strip() # .strip()で前後の余白を削除
        else:
            title_text = "（タイトル無し）"
        
        # 【課題要件】辞書に格納 (key: URL, value: title)
        sitemap_data[current_url] = title_text

        # 【課題要件】同一ドメインの全リンクを辿る
        # <a>タグ（href属性を持つもの）を全て探す
        # ※BeautifulSoupはコメントアウトされたタグを自動で無視します
        for link in soup.find_all('a', href=True):
            href = link['href']
            
            # /access などの相対パスを https://.../access のような絶対パスに変換
            next_url = urljoin(current_url, href)
            
            # #section などのアンカー（ページ内リンク）を削除
            next_url = next_url.split('#')[0]

            # 【課題要件】同一ドメインかチェック
            if urlparse(next_url).netloc == BASE_DOMAIN:
                # まだ訪問しておらず、これから訪問するリストにもなければ追加
                if next_url not in visited and next_url not in to_visit:
                    to_visit.add(next_url)

    except requests.RequestException as e:
        print(f"  > エラー: アクセスできませんでした ({e})")
    except Exception as e:
        print(f"  > 予期せぬエラー: {e}")

print("クローリングが完了しました。")

[32] チェック中: https://www.musashino-u.ac.jp/guide/information/iraction.html
[33] チェック中: http://www.musashino-u.ac.jp/international/international-students/
[34] チェック中: https://www.musashino-u.ac.jp/basic/endowment_course.html
[35] チェック中: https://www.musashino-u.ac.jp/admission/graduate_school/pastpapers.html
[36] チェック中: http://www.musashino-u.ac.jp/research/laboratory/center_for_clinical_pharmacy.html
[37] チェック中: http://www.musashino-u.ac.jp/academics/advanced_course/
[38] チェック中: http://www.musashino-u.ac.jp/student-life/tool/
[39] チェック中: http://www.musashino-u.ac.jp/student-life/campus_life/calendar.html
[40] チェック中: https://www.musashino-u.ac.jp/basic/buddhist_studies/
[41] チェック中: https://www.musashino-u.ac.jp/guide/facility/MUSIC_center/
[42] チェック中: http://www.musashino-u.ac.jp/international/international-students/study_abroad_agreement/japanese_language_course.html
[43] チェック中: https://www.musashino-u.ac.jp/academics/graduate_school/course/human_and_social_sciences/major/st.html
[44] チェ

KeyboardInterrupt: 

In [3]:
# 【課題要件】辞書型変数を print() で表示する
print("\n--- サイトマップ抽出結果 (30件分) ---")

# sitemap_data.items() で、key(url)とvalue(title)を同時に取り出す
for url, title in sitemap_data.items():
    print(f"URL: {url}")
    print(f"Title: {title}\n") # \n で1行空けて見やすくする

print(f"\n合計 {len(sitemap_data)} ページを抽出しました。")


--- サイトマップ抽出結果 (30件分) ---
URL: https://www.musashino-u.ac.jp/
Title: æ­¦èµéå¤§å­¦

URL: https://www.musashino-u.ac.jp/guide/campus/
Title: ã­ã£ã³ãã¹ | å¤§å­¦æ¡å | æ­¦èµéå¤§å­¦

URL: https://www.musashino-u.ac.jp/guide/activities/SDGs.html
Title: æ­¦èµéå¤§å­¦SDGså®è¡å®£è¨ | å¤§å­¦æ¡å | æ­¦èµéå¤§å­¦

URL: https://www.musashino-u.ac.jp/guide/facility/off-campus_learning_office.html
Title: å­¦å¤å­¦ä¿®æ¨é²ã»ã³ã¿ã¼ | å¤§å­¦æ¡å | æ­¦èµéå¤§å­¦

URL: https://www.musashino-u.ac.jp/research/laboratory/
Title: ç ç©¶æã»ç ç©¶å®¤ã»ã»ã³ã¿ã¼ | ç ç©¶ | æ­¦èµéå¤§å­¦

URL: https://www.musashino-u.ac.jp/research/laboratory/miga/
Title: å½éç·åç ç©¶æ | ç ç©¶ | æ­¦èµéå¤§å­¦

URL: https://www.musashino-u.ac.jp/guide/information/evaluation.html
Title: ææ¥­ãªãã¬ã¯ã·ã§ã³ | å¤§å­¦æ¡å | æ­¦èµéå¤§å­¦

URL: https://www.musashino-u.ac.jp/research/kakenhi/
Title: ç§å­¦ç ç©¶è²»å©æäºæ¥­ | ç ç©¶ | æ­¦èµéå¤§å­¦

URL: https://www.m